# Lecture 07 solutions — Forecasting with past information

These cells provide complete answers and executable calculations. Every score below is for rolling one-step forecasts unless explicitly stated otherwise.

## 1. Information at the origin

The conditional expectation is $\mathcal F_t$-measurable by definition of conditional expectation; it uses only events distinguishable from information available at origin $t$. Tomorrow's weekday is a deterministic calendar value known in advance. Tomorrow's realized sales are not known and cannot be a feature. The generated model can use past sales, lagged summaries, and calendar encodings.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def find_data_file(filename):
    """Search this directory and its parents for a file under data/."""
    for directory in (Path.cwd(), *Path.cwd().parents):
        candidate = directory / "data" / filename
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not find data/{filename} from {Path.cwd()}")


sales_data = pd.read_csv(find_data_file("daily_sales.csv"), parse_dates=["date"])
sales_data = sales_data.sort_values("date").reset_index(drop=True)
assert sales_data["date"].is_monotonic_increasing and sales_data["date"].is_unique
sales_data.head()

In [ ]:
def make_forecasting_features(data):
    """Build rolling one-step predictors using only observations before each date."""
    frame = data[["date", "sales"]].copy().sort_values("date").reset_index(drop=True)
    day = frame["date"].dt.dayofweek.to_numpy()
    frame["weekday_sin"] = np.sin(2 * np.pi * day / 7)
    frame["weekday_cos"] = np.cos(2 * np.pi * day / 7)
    frame["year_day_sin"] = np.sin(2 * np.pi * frame["date"].dt.dayofyear / 365.25)
    frame["year_day_cos"] = np.cos(2 * np.pi * frame["date"].dt.dayofyear / 365.25)
    for lag in (1, 2, 7, 14, 28):
        frame[f"lag_{lag}"] = frame["sales"].shift(lag)
    frame["mean_7_before"] = frame["sales"].shift(1).rolling(7).mean()
    frame["mean_28_before"] = frame["sales"].shift(1).rolling(28).mean()
    return frame.dropna().reset_index(drop=True)


feature_data = make_forecasting_features(sales_data)
feature_columns = [
    column for column in feature_data.columns if column not in ("date", "sales")
]
feature_data[["date", "sales", "lag_1", "lag_7", "mean_7_before", "weekday_sin"]].head()

## 2. Leakage-free lags

`shift(1)` moves the series so that the value at row $t$ is absent before its rolling window is computed. Thus the 7-day mean contains $Y_{t-7},\ldots,Y_{t-1}$ and not $Y_t$. The same rule applies to every historical statistic.

In [ ]:
row = feature_data.iloc[100]
assert np.isclose(row["lag_7"], sales_data.loc[121, "sales"])
assert np.isclose(row["mean_7_before"], sales_data.loc[121:127, "sales"].mean())
print(row[["date", "sales", "lag_7", "mean_7_before"]])

## Leakage property check

Change sales strictly after one origin. Its lag and rolling features at that earlier origin must stay the same.

In [ ]:
origin_raw_index = 200
origin_feature_index = origin_raw_index - 28
future_perturbation = sales_data.copy()
future_perturbation.loc[origin_raw_index + 1 :, "sales"] += 10000
perturbed_features = make_forecasting_features(future_perturbation)
predictor_columns = [
    column for column in feature_data.columns if column not in ("date", "sales")
]
np.testing.assert_allclose(
    feature_data.loc[origin_feature_index, predictor_columns].to_numpy(dtype=float),
    perturbed_features.loc[origin_feature_index, predictor_columns].to_numpy(
        dtype=float
    ),
)
print("Future perturbations leave origin features unchanged.")

## 3. Chronological split

The first 70% of usable rows train the model, the following 15% choose complexity, and the final 15% assess the selected model. Shuffling could put future dates in training while scoring earlier dates, and it breaks the simulation of an origin where only the past is available.

In [ ]:
n_rows = len(feature_data)
train_end = int(0.70 * n_rows)
validation_end = int(0.85 * n_rows)
train_data = feature_data.iloc[:train_end].copy()
validation_data = feature_data.iloc[train_end:validation_end].copy()
test_data = feature_data.iloc[validation_end:].copy()
boundaries = pd.DataFrame(
    {
        "partition": ["train", "validation", "test"],
        "first": [
            part.date.min().date() for part in (train_data, validation_data, test_data)
        ],
        "last": [
            part.date.max().date() for part in (train_data, validation_data, test_data)
        ],
    }
)
boundaries

## 4. Seasonal-naive performance

The prior same-weekday value encodes recurring weekly behavior and has no fitted parameters. Its score is a useful hurdle for a more flexible learner.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

feature_columns = [
    column for column in feature_data.columns if column not in ("date", "sales")
]
actual = test_data["sales"].to_numpy()
seasonal_prediction = test_data["lag_7"].to_numpy()
naive_scores = {
    "MAE": mean_absolute_error(actual, seasonal_prediction),
    "RMSE": mean_squared_error(actual, seasonal_prediction) ** 0.5,
}
naive_scores

## 5. Validation-selected XGBoost and test scores

We use the validation block to choose the iteration count via early stopping. Test outcomes are only evaluated after fitting.

In [ ]:
forecast_model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=500,
    learning_rate=0.035,
    max_depth=3,
    min_child_weight=5,
    reg_lambda=5.0,
    early_stopping_rounds=25,
    eval_metric="rmse",
    random_state=17,
    n_jobs=2,
)
forecast_model.fit(
    train_data[feature_columns],
    train_data["sales"],
    eval_set=[(validation_data[feature_columns], validation_data["sales"])],
    verbose=False,
)
model_prediction = forecast_model.predict(test_data[feature_columns])
score_comparison = pd.DataFrame(
    {
        "seasonal naive": naive_scores,
        "XGBoost": {
            "MAE": mean_absolute_error(actual, model_prediction),
            "RMSE": mean_squared_error(actual, model_prediction) ** 0.5,
        },
    }
).T
score_comparison["best_iteration"] = [np.nan, forecast_model.best_iteration]
score_comparison

The score is rolling one-step: after each test target is observed, it can enter later lag windows. Model parameters remain fixed. In a fixed-origin 14-day forecast, no actual from within the 14-day horizon is available; recursively insert prior predictions into lag features, or fit direct horizon-specific models.

## 6. Residual plot

A residual is actual minus predicted. Runs above or below zero suggest temporal structure remains unexplained; the plot should be read alongside aggregate scores.

In [ ]:
residuals = actual - model_prediction
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True, layout="constrained")
axes[0].plot(test_data["date"], actual, label="actual", color="black")
axes[0].plot(test_data["date"], model_prediction, label="XGBoost")
axes[0].legend(frameon=False)
axes[1].scatter(test_data["date"], residuals, s=12)
axes[1].axhline(0, color="black", lw=1)
axes[1].set(ylabel="actual - predicted", xlabel="date")
fig.autofmt_xdate()
plt.show()

## 7. Evaluation design

The model is trained before the test block; at each test date it issues a one-day-ahead forecast, updates lag features with the latest realized sales, and keeps its fitted parameters fixed. A fixed-origin 14-day assessment requires forecasts made from one origin, recursive predicted sales or horizon-specific outputs throughout the horizon, and no actual outcomes from that period.